PIFOCast, a simple weather prediction model
===========================================


##### Copyright 2025 Nicolas Gasnier.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

##### Introduction

This project is based on the following :
- Google Deepmind [GraphCast](https://www.science.org/doi/10.1126/science.adi2336) and [GenCast](https://arxiv.org/abs/2312.15796) papers, and associated [source code](https://github.com/google-deepmind/graphcast) as a source of inspiration
- Tensorflow GNN's colab example ["Learning shortest path with GraphNetowks in TF-GNN"](https://colab.research.google.com/github/tensorflow/gnn/blob/master/examples/notebooks/graph_network_shortest_path.ipynb#scrollTo=qr1_8ttC08vu), from wich the base Encoder / Processor / Decoder architecture is derived.


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import collections
import functools
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import runner
from typing import Callable, Optional, Mapping

import importlib

# Theses two rows for improving development workflow of our module
import pifocast
importlib.reload(pifocast)

from pifocast import LatLonGrid, pifo, buildGridGNN, getGraphExample, getGraphForFeatures, get_dataset, load_dataset, pifoGridGenerator


### Dataset generation - building the whole dataset from Era5

In [2]:
schema = tfgnn.read_schema("pifo.pbtxt")
graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
grid = LatLonGrid("dataset/202504.grib", 4)



2025-06-02 19:19:31.676397: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
with tf.io.TFRecordWriter("dataset/inputDS.tfrecord") as inputDSWriter, tf.io.TFRecordWriter("dataset/targetDS.tfrecord") as targetDSWriter:
    for graph, target in pifoGridGenerator("dataset/202504.grib", grid):
        #graph, target = getGraphExample(grid)
        example = tfgnn.write_example(graph)
        inputDSWriter.write(example.SerializeToString())
        targetSR = tf.io.serialize_tensor(target)
        targetFeature = {
            'target' : tf.train.Feature(bytes_list=tf.train.BytesList(value=[targetSR.numpy()]))
        }    
        targetExample = tf.train.Example(features=tf.train.Features(feature=targetFeature))
        targetDSWriter.write(targetExample.SerializeToString())



In [4]:
trainDS = load_dataset(graph_spec, "dataset/inputDS.tfrecord", "dataset/targetDS.tfrecord")



In [5]:
print("listing...")
#print(next(iter(graphDS)))
for i, graph in enumerate(trainDS.batch(2).take(3)):
     print(f"Input {i}: {graph}")

listing...
Input 0: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']), <tf.Tensor: shape=(2, 65160, 3), dtype=float32, numpy=
array([[[0.34887853, 0.37536436, 0.50481313],
        [0.34887853, 0.37536436, 0.50481313],
        [0.34887853, 0.37536436, 0.50481313],
        ...,
        [0.24723761, 0.3927586 , 0.4014648 ],
        [0.24723761, 0.3927586 , 0.4014648 ],
        [0.24723761, 0.3927586 , 0.4014648 ]],

       [[0.34462795, 0.38200247, 0.4688942 ],
        [0.34462795, 0.38200247, 0.4688942 ],
        [0.34462795, 0.38200247, 0.4688942 ],
        ...,
        [0.2575431 , 0.36480495, 0.43032122],
        [0.2575431 , 0.36480495, 0.43032122],
        [0.2575431 , 0.36480495, 0.43032122]]], dtype=float32)>)


2025-06-02 19:19:51.388119: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144


Input 1: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']), <tf.Tensor: shape=(2, 65160, 3), dtype=float32, numpy=
array([[[0.34008452, 0.40168437, 0.47430304],
        [0.34008452, 0.40168437, 0.47430304],
        [0.34008452, 0.40168437, 0.47430304],
        ...,
        [0.2527301 , 0.30659792, 0.4688879 ],
        [0.2527301 , 0.30659792, 0.4688879 ],
        [0.2527301 , 0.30659792, 0.4688879 ]],

       [[0.3392722 , 0.41207486, 0.45809186],
        [0.3392722 , 0.41207486, 0.45809186],
        [0.3392722 , 0.41207486, 0.45809186],
        ...,
        [0.25037232, 0.38631955, 0.46661952],
        [0.25037232, 0.38631955, 0.46661952],
        [0.25037232, 0.38631955, 0.46661952]]], dtype=float32)>)
Input 2: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']), <tf.Tensor: shap

2025-06-02 19:19:51.589944: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
